### Бинарная мягкая классификация правдивости новости через Random Forest

In [13]:
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import label_binarize


In [15]:
train_data = load_npz("x_train_tfidf.npz")
test_data = load_npz("x_test_tfidf.npz")

X_train = train_data[:, :-1]
y_train = np.ravel(train_data[:, -1].toarray()).astype(int)

X_test = test_data[:, :-1]
y_test = np.ravel(test_data[:, -1].toarray()).astype(int)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Количество классов:", len(np.unique(y_train)))


X_train: (57707, 50000)
X_test: (14427, 50000)
Количество классов: 2


In [17]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    random_state=67,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]

print("test roc-auc:", roc_auc_score(y_test, proba))
print("test pr-auc:", average_precision_score(y_test, proba))

test roc-auc: 0.9935847334044398
test pr-auc: 0.9939445073090226


In [252]:
from sklearn.feature_extraction.text import TfidfVectorizer

test_news = [
    """
    The White House announced a new economic plan on Tuesday aimed at supporting small businesses,
    reducing inflationary pressure, and expanding access to affordable loans for local manufacturers.
    According to administration officials, the proposal includes tax credits for companies that invest
    in domestic production, additional funding for infrastructure projects, and measures designed to
    lower supply chain costs. The plan is expected to be reviewed by Congress next month, where lawmakers
    from both parties are likely to debate the size of the spending package and its potential impact on
    the federal budget. Economists said the proposal could provide short-term support for businesses,
    although its long-term effect will depend on implementation and broader market conditions.
    """,

    """
    BREAKING: Secret documents leaked by anonymous insiders reveal that top government officials have
    been hiding a massive plan to control every smartphone in America using invisible signals from new
    wireless towers. According to the report, the technology can allegedly read private messages, track
    thoughts, and silently change people’s opinions without their knowledge. The shocking discovery was
    supposedly confirmed by unnamed experts who claim that former presidents, major technology companies,
    and international organizations are all involved in the operation. Officials have refused to confirm
    the story, which supporters say proves that the truth is being covered up by the media.
    """
]

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
)

news_tfidf = tfidf.transform(test_news)
preds = rf.predict(news_tfidf)
probas = rf.predict_proba(news_tfidf)

prediction_table = pd.DataFrame({
    "text": test_news,
    "fake_probability": probas[:, 1],
})

prediction_table


,text,fake_probability
0,\n The White House announced a new economic...,0.316706
1,\n BREAKING: Secret documents leaked by ano...,0.860777


### Многоклассовая мягкая классификация категорий новостей через Random Forest

In [4]:
datacat = pd.read_json("data/News_Category_Dataset_v3.json", lines=True)
category_names = np.sort(datacat["category"].unique())
category_names


array(['ARTS', 'ARTS & CULTURE', 'BLACK VOICES', 'BUSINESS', 'COLLEGE',
       'COMEDY', 'CRIME', 'CULTURE & ARTS', 'DIVORCE', 'EDUCATION',
       'ENTERTAINMENT', 'ENVIRONMENT', 'FIFTY', 'FOOD & DRINK',
       'GOOD NEWS', 'GREEN', 'HEALTHY LIVING', 'HOME & LIVING', 'IMPACT',
       'LATINO VOICES', 'MEDIA', 'MONEY', 'PARENTING', 'PARENTS',
       'POLITICS', 'QUEER VOICES', 'RELIGION', 'SCIENCE', 'SPORTS',
       'STYLE', 'STYLE & BEAUTY', 'TASTE', 'TECH', 'THE WORLDPOST',
       'TRAVEL', 'U.S. NEWS', 'WEDDINGS', 'WEIRD NEWS', 'WELLNESS',
       'WOMEN', 'WORLD NEWS', 'WORLDPOST'], dtype=object)

In [5]:
train_data = load_npz("x_train_cat_tfidf.npz")
test_data = load_npz("x_test_cat_tfidf.npz")

X_train = train_data[:, :-1]
y_train = np.ravel(train_data[:, -1].toarray()).astype(int)

X_test = test_data[:, :-1]
y_test = np.ravel(test_data[:, -1].toarray()).astype(int)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Количество классов:", len(np.unique(y_train)))


X_train: (167621, 50000)
X_test: (41906, 50000)
Количество классов: 42


In [6]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features="sqrt",
    random_state=67,
    n_jobs=-1,
)

rf.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [7]:
y_proba = rf.predict_proba(X_test)
classes = rf.classes_
y_test_bin = label_binarize(y_test, classes=classes)


In [8]:
macro_roc_auc = roc_auc_score(
    y_test_bin,
    y_proba,
    average="macro",
    multi_class="ovr",
)

micro_roc_auc = roc_auc_score(
    y_test_bin,
    y_proba,
    average="micro",
    multi_class="ovr",
)

macro_pr_auc = average_precision_score(
    y_test_bin,
    y_proba,
    average="macro",
)

micro_pr_auc = average_precision_score(
    y_test_bin,
    y_proba,
    average="micro",
)

print("Macro ROC-AUC:", macro_roc_auc)
print("Micro ROC-AUC:", micro_roc_auc)
print("Macro PR-AUC:", macro_pr_auc)
print("Micro PR-AUC:", micro_pr_auc)


Macro ROC-AUC: 0.9259525566613578
Micro ROC-AUC: 0.9505335972169565
Macro PR-AUC: 0.42646822275379476
Micro PR-AUC: 0.586010259307963


In [12]:
class_metrics = []

for class_index, class_id in enumerate(classes):
    class_name = category_names[class_id]

    class_metrics.append({
        "category": class_name,
        "roc_auc": roc_auc_score(y_test_bin[:, class_index], y_proba[:, class_index]),
        "pr_auc": average_precision_score(y_test_bin[:, class_index], y_proba[:, class_index]),
    })

class_metrics = pd.DataFrame(class_metrics)
class_metrics.sort_values("pr_auc", ascending=False)


,category,roc_auc,pr_auc
36,WEDDINGS,0.989879,0.831326
24,POLITICS,0.951554,0.826348
30,STYLE & BEAUTY,0.973160,0.819506
8,DIVORCE,0.973081,0.728752
25,QUEER VOICES,0.949995,0.725369
17,HOME & LIVING,0.963423,0.712153
34,TRAVEL,0.960167,0.697357
38,WELLNESS,0.945330,0.674876
13,FOOD & DRINK,0.973396,0.674864
10,ENTERTAINMENT,0.939100,0.673075
